In [0]:
%run ./00_config_and_logging

In [0]:
from pyspark.sql.functions import (
    col, lit, expr, current_timestamp, to_date,
    row_number, when, count, isnull
)
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
# Read NVD bronze — rename 'id' → 'cve_id'
nvd_bronze = spark.table(BRONZE_NVD).withColumnRenamed('id','cve_id')
# Read EPSS bronze — rename 'cve' → 'cve_id'
epss_bronze = spark.table(BRONZE_EPSS).withColumnRenamed('cve', 'cve_id')
# Read KEV bronze — rename 'cveID' → 'cve_id'
kev_bronze = spark.table(BRONZE_KEV).withColumnRenamed('cveID','cve_id')

In [0]:
w_nvd = Window.partitionBy('cve_id').orderBy(col('load_timestamp').desc())
nvd_dedup = nvd_bronze.withColumn('rn', row_number().over(w_nvd)) \
    .filter(col('rn') == 1).drop('rn')
w_epss = Window.partitionBy('cve_id').orderBy(col('load_timestamp').desc())
epss_dedup = epss_bronze.withColumn('rn', row_number().over(w_epss)) \
    .filter(col('rn') == 1).drop('rn')
w_kev = Window.partitionBy('cve_id').orderBy(col('load_timestamp').desc())
kev_dedup = kev_bronze.withColumn('rn', row_number().over(w_kev)) \
    .filter(col('rn') == 1).drop('rn')

In [0]:
nvd_flat = nvd_dedup.select(
    "cve_id",
    # English description — filter the descriptions array for lang='en'
    expr("filter(descriptions, x -> x.lang = 'en')[0]['value']").alias("description"),
    # CVSS v3.1 (preferred) — extract from metrics.cvssMetricV31[0].cvssData
    col("metrics.cvssMetricV31")[0]["cvssData"]["baseScore"].alias("cvss_v31_score"),
    col("metrics.cvssMetricV31")[0]["cvssData"]["baseSeverity"].alias("cvss_v31_severity"),
    col("metrics.cvssMetricV31")[0]["cvssData"]["vectorString"].alias("cvss_v31_vector"),
    # CVSS v3.0 (fallback)
    col("metrics.cvssMetricV30")[0]["cvssData"]["baseScore"].alias("cvss_v30_score"),
    col("metrics.cvssMetricV30")[0]["cvssData"]["baseSeverity"].alias("cvss_v30_severity"),
    # CVSS v2 (last resort)
    col("metrics.cvssMetricV2")[0]["cvssData"]["baseScore"].alias("cvss_v2_score"),
    col("metrics.cvssMetricV2")[0]["baseSeverity"].alias("cvss_v2_severity"),
    # CWE IDs — flatten nested weaknesses[].description[].value
    expr("transform(flatten(transform(weaknesses, w -> w.description)), d -> d.value)").alias("cwe_ids"),
    col("vulnStatus").alias("vuln_status"),
    col("published").alias("published_date"),
    col("lastModified").alias("last_modified"),
    col("sourceIdentifier").alias("source_identifier"),
)

In [0]:
from pyspark.sql.functions import coalesce

nvd_flat = nvd_flat \
    .withColumn('cvss_base_score',
        coalesce(
            col('cvss_v31_score'),
            col('cvss_v30_score'),
            col('cvss_v2_score')
        )
    ) \
    .withColumn('cvss_severity',
        coalesce(
            col('cvss_v31_severity'),
            col('cvss_v30_severity'),
            col('cvss_v2_severity')
        )
    ) \
    .withColumn('cvss_vector',
        coalesce(
            col('cvss_v31_vector'),
        )
    )
#

In [0]:
kev_clean = kev_dedup.select(
    "cve_id",
    to_date(col("dateAdded"), "yyyy-MM-dd").alias("kev_date_added"),
    to_date(col("dueDate"), "yyyy-MM-dd").alias("kev_due_date"),
    col("vendorProject").alias("vendor_project"),
    col("product"),
    col("vulnerabilityName").alias("vulnerability_name"),
    col("shortDescription").alias("short_description"),
    col("requiredAction").alias("required_action"),
    col("knownRansomwareCampaignUse").alias("known_ransomware_use"),
    col("cwes").alias("kev_cwes"),
    col("notes").alias("kev_notes"),
)

In [0]:
epss_clean = epss_dedup.select(
    "cve_id",
    col("epss").alias("epss_score"),
    col("percentile").alias("epss_percentile"),
)

In [0]:
silver_unified = nvd_flat \
    .join(epss_clean, "cve_id", "left") \
    .join(kev_clean, "cve_id", "left") \
    .withColumn("is_in_kev", when(col("kev_date_added").isNotNull(), lit(True)).otherwise(lit(False))) \
    .withColumn("silver_load_timestamp", current_timestamp())

print(f"Unified silver rows: {silver_unified.count()}")
silver_unified.printSchema()
display(silver_unified.limit(20))

In [0]:
# 1. Duplicate CVE IDs in NVD bronze (should be 0 after dedup)
nvd_dupes = nvd_dedup.groupBy("cve_id").count().filter("count > 1")
print(f"NVD duplicate cve_ids: {nvd_dupes.count()}")

# 2. CVEs in KEV but NOT in NVD (orphaned threats)
kev_only = kev_dedup.select("cve_id").subtract(nvd_dedup.select("cve_id"))
print(f"KEV CVEs missing from NVD: {kev_only.count()}")
if kev_only.count() > 0:
    kev_only.show(20, truncate=False)

# 3. CVEs in EPSS but NOT in NVD
epss_only = epss_dedup.select("cve_id").subtract(nvd_dedup.select("cve_id"))
print(f"EPSS CVEs missing from NVD: {epss_only.count()}")

# 4. Null descriptions in NVD
null_descs = nvd_flat.filter(col("description").isNull()).count()
print(f"NVD rows with null description: {null_descs}")

# 5. Missing CVSS base score (no v3.1, v3.0, or v2)
no_cvss = nvd_flat.filter(col("cvss_base_score").isNull()).count()
print(f"NVD rows with no CVSS score: {no_cvss}")

# 6. KEV entries with overdue due dates (due date in the past)
from pyspark.sql.functions import current_date
overdue = kev_clean.filter(col("kev_due_date") < current_date())
print(f"Overdue KEV entries: {overdue.count()}")
if overdue.count() > 0:
    overdue.select("cve_id", "kev_due_date").show(10, truncate=False)

In [0]:
# print(f"Unified silver rows: {silver_unified.count()}")
# null_values = silver_unified.filter(col('kev_date_added').isNull()).count()
# print(f"Null kev rows: {null_values}")


print(f"Unified silver rows: {silver_unified.count()}")
valid_kev_vals = silver_unified.filter(col('kev_date_added').isNotNull()).count()
print(f"Valid kev rows: {valid_kev_vals} Total Possible : 1181")

valid_epss_vals = silver_unified.filter(col('epss_score').isNotNull()).count()
print(f"Valid epss rows: {valid_epss_vals} Total Possible : 249782")

rejected_count = silver_unified.filter(col('vuln_status') =='Rejected').count()
print(f"Rejected rows: {rejected_count}")

print(f'CVEs with only cve scores : {silver_unified.count()-rejected_count-valid_epss_vals}')